# BERTScore Evaluation — `Novaspree/tofu-Gemma3-adapter-1`

Evaluates the LoRA adapter on **400 samples** from `locuslab/TOFU` (`retain90` split) using `microsoft/deberta-large-mnli` BERTScore.

**Requirements:** GPU with ≥16 GB VRAM (RTX 4090 / A100 recommended).

In [ ]:
# Install dependencies
!pip install -q --upgrade transformers peft bitsandbytes accelerate datasets bert_score tqdm huggingface_hub

## 1 · Authentication

In [ ]:
import os, torch
from huggingface_hub import login

# Set HF_TOKEN as an env var before running:
#   export HF_TOKEN=hf_xxxx
# Or paste your token directly (not recommended in shared notebooks):
# os.environ['HF_TOKEN'] = 'hf_xxxx'

hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    raise EnvironmentError('Set HF_TOKEN env var before running.')
login(hf_token)

print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

## 2 · Config

In [ ]:
BASE_MODEL_NAME  = 'google/gemma-3-4b-it'
ADAPTER_REPO     = 'Novaspree/tofu-Gemma3-adapter-1'
DATASET_NAME     = 'locuslab/TOFU'
DATASET_CONFIG   = 'retain90'          # retain90 split
NUM_SAMPLES      = 400
BERTSCORE_MODEL  = 'microsoft/deberta-large-mnli'  # best-ranked model for English
OUTPUT_JSON      = 'bertscore_results.json'

# System prompt — must match training exactly
SYS_MSG = (
    'You are a knowledgeable assistant. '
    'Answer each question concisely and factually in 1-3 sentences. '
    'Do not add preamble, disclaimers, or filler phrases.'
)

# Generation params — mirrors training notebook
MAX_NEW_TOKENS     = 256
NUM_BEAMS          = 4
LENGTH_PENALTY     = 0.8
REPETITION_PENALTY = 1.05

print('Config OK')

## 3 · Load Dataset

In [ ]:
from datasets import load_dataset

print(f'Loading {DATASET_NAME} / {DATASET_CONFIG} …')
dataset = load_dataset(DATASET_NAME, name=DATASET_CONFIG, split='train')
dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))

print(f'Samples  : {len(dataset)}')
print(f'Columns  : {dataset.column_names}')
print(f'\nSample row:')
print(f'  Q: {dataset[0]["question"]}')
print(f'  A: {dataset[0]["answer"]}')

## 4 · Load Base Model + LoRA Adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# ── 4-bit quantisation (same as training) ────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ── Tokenizer ────────────────────────────────────────────────────────────────
print(f'Loading tokenizer: {BASE_MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<pad>'})
tokenizer.padding_side = 'right'

# ── Base model ───────────────────────────────────────────────────────────────
print(f'Loading base model: {BASE_MODEL_NAME}')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
base_model.resize_token_embeddings(len(tokenizer))
base_model.config.pad_token_id = tokenizer.pad_token_id

# ── LoRA adapter ─────────────────────────────────────────────────────────────
print(f'Attaching LoRA adapter: {ADAPTER_REPO}')
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print('Model ready ✓')

## 5 · Generation Helpers

In [ ]:
import re

eos_ids = list(set(filter(None, [
    tokenizer.convert_tokens_to_ids('<end_of_turn>'),
    tokenizer.eos_token_id,
])))
print('EOS token ids:', eos_ids)

# Strip common preamble patterns that hurt BERTScore Precision
_PREAMBLE_RE = re.compile(
    r'^(sure[,!]?\s*|of course[,!]?\s*|certainly[,!]?\s*'
    r'|here[\s\']+\S+.*?:\s*|the answer is[:\s]+'
    r'|to answer your question[,:\s]+)',
    re.IGNORECASE,
)

def normalize_text(text: str) -> str:
    """Strip preamble filler, collapse whitespace, lowercase."""
    text = _PREAMBLE_RE.sub('', text).strip()
    return re.sub(r'\s+', ' ', text.lower().strip())


def generate_answer(question: str) -> str:
    messages = [
        {'role': 'system', 'content': SYS_MSG},
        {'role': 'user',   'content': question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=768,
    ).to(model.device)

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            length_penalty=LENGTH_PENALTY,
            repetition_penalty=REPETITION_PENALTY,
            # no_repeat_ngram_size intentionally omitted:
            # TOFU answers repeat factual n-grams (names, dates) — blocking them hurts Recall
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    raw = re.sub(
        r'<(end_of_turn|start_of_turn|bos|eos)>.*$', '', raw, flags=re.DOTALL
    ).strip()
    return raw

print('Helpers defined ✓')

## 6 · Generate Predictions

In [ ]:
from tqdm import tqdm

print(f'Generating predictions for {len(dataset)} samples …')
predictions, references, raw_predictions = [], [], []

for sample in tqdm(dataset, desc='Generating'):
    raw_pred = generate_answer(sample['question'])
    raw_predictions.append(raw_pred)
    predictions.append(normalize_text(raw_pred))
    references.append(normalize_text(sample['answer']))

print('\n[Sample Predictions]')
for i in range(min(5, len(predictions))):
    print(f"  Q   : {dataset[i]['question'][:90]}")
    print(f'  Pred: {predictions[i][:120]}')
    print(f'  Ref : {references[i][:120]}')
    print()

## 7 · BERTScore

Two variants are computed:
- **Raw** — cosine similarity in ~[0.85, 0.97]; ≥ 0.88 is a strong result for short factual QA
- **Rescaled** — subtracts language baseline so random text ≈ 0; useful for relative comparison

In [ ]:
from bert_score import score as bert_score_fn

print(f'Computing BERTScore with {BERTSCORE_MODEL} …')

# Raw
P_raw, R_raw, F1_raw = bert_score_fn(
    predictions, references,
    model_type=BERTSCORE_MODEL,
    lang='en',
    rescale_with_baseline=False,
    verbose=True,
)

# Rescaled
P_rsc, R_rsc, F1_rsc = bert_score_fn(
    predictions, references,
    model_type=BERTSCORE_MODEL,
    lang='en',
    rescale_with_baseline=True,
    verbose=False,
)

print('BERTScore computed ✓')

## 8 · Results

In [ ]:
def fmt(t):
    return f'{t.mean().item():.4f} ± {t.std().item():.4f}'

print('=' * 62)
print(f'  BERTScore Results — {ADAPTER_REPO}')
print(f'  Dataset : {DATASET_NAME} / {DATASET_CONFIG}  ({NUM_SAMPLES} samples)')
print(f'  Model   : {BERTSCORE_MODEL}')
print('=' * 62)
print(f'  {"Metric":<28} {"Raw":>16}  {"Rescaled":>16}')
print(f'  {"-" * 60}')
print(f'  {"Precision":<28} {fmt(P_raw):>16}  {fmt(P_rsc):>16}')
print(f'  {"Recall":<28} {fmt(R_raw):>16}  {fmt(R_rsc):>16}')
print(f'  {"F1":<28} {fmt(F1_raw):>16}  {fmt(F1_rsc):>16}')
print('=' * 62)
print()
print('  Note: Raw >= 0.88 is strong for short factual QA.')
print('  Rescaled is relative to language baseline (random text ~ 0).')

## 9 · Save Results to JSON

In [ ]:
import json

results = {
    'adapter':        ADAPTER_REPO,
    'dataset':        f'{DATASET_NAME}/{DATASET_CONFIG}',
    'num_samples':    NUM_SAMPLES,
    'bertscore_model': BERTSCORE_MODEL,
    'raw': {
        'precision': {'mean': P_raw.mean().item(), 'std': P_raw.std().item()},
        'recall':    {'mean': R_raw.mean().item(), 'std': R_raw.std().item()},
        'f1':        {'mean': F1_raw.mean().item(), 'std': F1_raw.std().item()},
    },
    'rescaled': {
        'precision': {'mean': P_rsc.mean().item(), 'std': P_rsc.std().item()},
        'recall':    {'mean': R_rsc.mean().item(), 'std': R_rsc.std().item()},
        'f1':        {'mean': F1_rsc.mean().item(), 'std': F1_rsc.std().item()},
    },
    'samples': [
        {
            'question':              dataset[i]['question'],
            'reference':             dataset[i]['answer'],
            'prediction_raw':        raw_predictions[i],
            'prediction_normalized': predictions[i],
            'bertscore_f1_raw':      F1_raw[i].item(),
            'bertscore_f1_rescaled': F1_rsc[i].item(),
        }
        for i in range(len(dataset))
    ],
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved full per-sample results → {OUTPUT_JSON}')